In [1]:
%load_ext autotime

import json
from pathlib import Path
import re
import time
from tqdm.auto import tqdm

import pandas as pd

from schema_discovery import (
    load_prompt,
    analyze_snapshot,
    process_response,
)
from data_snapshot.utils import load_json

# Inputs

In [2]:
# Inputs
SYSTEM_PROMPT_PATH = "prompts/field_labeling_system.md"
USER_PROMPT_PATH = "prompts/field_labeling_user.md"
MODEL_NAME = "gpt-5.4-mini"
SLEEP_SECONDS = 0.2  # safety throttle
OUTPUT_PATH = f"data/field_labeling_results.jsonl"

# Load data

In [3]:
system_prompt = load_prompt(SYSTEM_PROMPT_PATH)
user_prompt_template = load_prompt(USER_PROMPT_PATH)
df = pd.read_csv("outputs/ontology_v1.csv")

# Render ontology markdown

In [4]:
ontology_template = """### {CANONICAL_NAME}

Definition:
{DEFINITION}

Examples:
{EXAMPLES}"""

ontologies = {}
for _, row in df.iterrows():
    example_str = "\n".join(["- " + x.strip() for x in row["sample_fields"].split(",")])

    if row["status"] == "keep":
        replace_dict = {
            "{CANONICAL_NAME}": row["canonical_name"],
            "{DEFINITION}": row["definition"],
            "{EXAMPLES}": example_str,
        }
    
        text = ontology_template + "\n"
        for old, new in replace_dict.items():
            text = text.replace(old, new)

        ontologies[row["canonical_name"]] = text

    elif row["status"] == "merge":
        # Do not create but add examples
        text = ontologies[row["merge_into"]]
        text += example_str + "\n"
        ontologies[row["merge_into"]] = text
        
ontology_md = "\n---\n\n".join(ontologies.values())

In [5]:
print(ontology_md)

### title

Definition:
The displayed or inferred title, heading, or caption that identifies the snapshot.

Examples:
- table_title
- figure_title
- snapshot_title
- chart_title
- map_title
- table_caption

---

### subject_summary

Definition:
A concise description of the main analytical subject or purpose of the snapshot.

Examples:
- table_subject
- map_subject
- table_topic
- analytical_focus
- table_purpose
- assessment_topic

---

### subject_domain

Definition:
Broad topical, policy, sectoral, or thematic domains represented by the snapshot.

Examples:
- subject_domain
- subject_topic
- thematic_domain
- subject_area
- topic_domain
- policy_domain
- sector

---

### visualization_type

Definition:
The structural form used to present the data, such as table, map, line chart, bar chart, dashboard, or composite figure.

Examples:
- visualization_type
- map_type
- chart_purpose
- statistical_output_type
- financial_data_type

---

### language

Definition:
The language used in the sn

# Main pipeline

In [ ]:
# Build skip list
to_skip = set()
if Path(OUTPUT_PATH).exists():
    with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            to_skip.add(json.loads(line)["snapshot_id"])

In [ ]:
# Initialize dir
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_PATH, "a", encoding="utf-8") as out_f:
    for s in tqdm(snapshot_files):
        snapshot_id = s.name

        # Skip if in skip list
        if snapshot_id in to_skip:
            print(f"Skipped: {s.name}")
            continue

        # Base row — every row has the same keys
        row = {
            "snapshot_id": snapshot_id,
            "source": SOURCE,
            "model": MODEL_NAME,
            "raw_output": None,
            "usage": None,
            "cost": None,
            "status": None,
            "incomplete_details": None,
            "parsed_output": None,
            "error": None,
        }

        try:
            # Infer metadata file from snapshot filename
            parts = re.split("(_figure|_table)", s.stem)
            fname = parts[0]
            label = parts[1].lstrip("_")
            metadata_path = metadata_lookup.get(fname)
            if metadata_path:
                metadata = load_json(metadata_path)
            else:
                metadata = None

            # Create user prompt
            user_prompt = user_prompt_template.replace(
                "{DOCUMENT_METADATA_JSON}",
                json.dumps(metadata, indent=2, ensure_ascii=False),
            )

            # # Responses API
            # response = analyze_snapshot(
            #     system_prompt=system_prompt,
            #     user_prompt=user_prompt,
            #     image_path=str(s),
            #     model=MODEL_NAME,
            #     max_output_tokens=5000,
            # )

            # # Parse and process response
            # result = process_response(response, model=MODEL_NAME)

            # Log data
            row["raw_output"] = result["raw_output"]
            row["usage"] = result["usage"]
            row["cost"] = result["cost"]
            row["status"] = result["status"]
            row["incomplete_details"] = result["incomplete_details"]
            row["parsed_output"] = result["parsed_output"]
            row["error"] = result["error"]  # None on success, message on parse failure

        except Exception as e:
            row["error"] = str(e)

        # Single write point — always executes
        out_f.write(json.dumps(row, ensure_ascii=False) + "\n")
        out_f.flush()

        time.sleep(SLEEP_SECONDS)